# Setup

- macOS: Run 'brew install poppler'
- Ubuntu/Debian: Run 'apt-get install poppler-utils'
- Windows: Install from https://github.com/oschwartz10612/poppler-windows/releases/ and add to PATH

In [ ]:
# !pip install typhoon-ocr==0.4.1
# !pip install python-dotenv
# !pip install lxml
# !pip install html5lib

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
from typhoon_ocr import ocr_document
from io import StringIO
import re

import sys
import os

sys.path.append(os.path.abspath(".."))
from utils import *

load_dotenv()

In [ ]:
KEYWORD = "partylist"

PARTYLIST_PAGE_I_1 = 10
PARTYLIST_PAGE_I_2 = 24
PARTYLIST_PAGE_I_3 = 23

PARTICIPANTS_PARTYLIST = [
    "ไทยทรัพย์ทวี", "เพื่อชาติไทย", "ใหม่",
    "มิติใหม่", "รวมใจไทย", "รวมไทยสร้างชาติ",
    "พลวัต", "ประชาธิปไตยใหม่", "เพื่อไทย",
    "ทางเลือกใหม่",

    "เศรษฐกิจ", "เสรีรวมไทย", "รวมพลังประชาชน", 
    "ท้องที่ไทย", "อนาคตไทย", "พลังเพื่อไทย", 
    "ไทยชนะ", "พลังสังคมใหม่", "สังคมประชาธิปไตยไทย", 
    "ฟิวชัน", "ไทรวมพลัง", "ก้าวอิสระ", 
    "ปวงชนไทย", "วิชชั่นใหม่", "เพื่อชีวิตใหม่", 
    "คลองไทย", "ประชาธิปัตย์", "ไทยก้าวหน้า", 
    "ไทยภักดี", "แรงงานสร้างชาติ", "ประชากรไทย", 
    "ครูไทยเพื่อประชาชน", "ประชาชาติ", "สร้างอนาคตไทย",

    "รักชาติ", "ไทยพร้อม",
    "ภูมิใจไทย", "พลังธรรมใหม่", "กรีน",
    "ไทยธรรม", "แผ่นดินธรรม", "กล้าธรรม",
    "พลังประชารัฐ", "โอกาสใหม่", "เป็นธรรม",
    "ประชาชน", "ประชาไทย", "ไทยสร้างไทย",
    "ไทยก้าวใหม่", "ประชาอาสาชาติ", "พร้อม",
    "เครือข่ายชาวนาแห่งประเทศไทย", "ไทยพิทักษ์ธรรม", "ความหวังใหม่",
    "ไทยรวมไทย", "เพื่อบ้านเมือง", "พลังไทยรักชาติ",
]

# OCR

In [ ]:
API_KEY = os.getenv("TYPHOON_OCR_API_KEY")

### Reader Function

In [ ]:
# อ่านข้อมูล file ตระกูลสส.บัญชีรายชื่อ
def read_partylist(file_name: str, page_num: int):
    print(f"Processing Partylist - File: {file_name} - Page: {page_num}-{page_num+2}")
    safe_path = get_safe_pdf_path(file_name)
    
    # Page i
    data_json_page_i_1 = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=safe_path,
        page_num=page_num,
        target_image_dim=2400,
    )
    
    # Page i+1
    data_json_page_i_2 = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=safe_path,
        page_num=page_num+1,
        target_image_dim=2400,
    )
    
    # Page i+2
    data_json_page_i_3 = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=safe_path,
        page_num=page_num+2,
        target_image_dim=2400,
    )
    
    # print(f"Page {page_num}")
    data_json_page_i_1 = thai_to_arabic(data_json_page_i_1)
    df_page_i_1 = pd.read_html(StringIO(data_json_page_i_1))[0]
    
    # print(f"Page {page_num+1}")
    data_json_page_i_2 = thai_to_arabic(data_json_page_i_2)
    df_page_i_2 = pd.read_html(StringIO(data_json_page_i_2))[0]
    
    # print(f"Page {page_num+2}")
    data_json_page_i_3 = thai_to_arabic(data_json_page_i_3)
    df_page_i_3 = pd.read_html(StringIO(data_json_page_i_3))[0]

    all_card = re.search(
        r'(?:ได้รับบัตรเลือกตั้ง.*?ทั้งหมด|จำนวนบัตรเลือกตั้งที่ได้รับจัดสรร.*?จำนวน)\s*(\d[\d,]*)\s*บัตร',
        data_json_page_i_1
    )
    good_card = re.search(r'บัตรดี.*?(\d[\d,]*)\s*บัตร', data_json_page_i_1)
    bad_card = re.search(r'บัตรเสีย.*?(\d[\d,]*)\s*บัตร', data_json_page_i_1)
    none_card = re.search(r'บัตรที่ไม่เลือกบัญชีรายชื่อของพรรคการเมืองใด.*?(\d[\d,]*)\s*บัตร', data_json_page_i_1)

    summary = {
        'จำนวนบัตรทั้งหมด': parse_number(all_card.group(1)) if all_card else 0,
        'บัตรดี': parse_number(good_card.group(1)) if good_card else 0,
        'บัตรเสีย': parse_number(bad_card.group(1)) if bad_card else 0,
        'ไม่เลือกผู้ใด': parse_number(none_card.group(1)) if none_card else 0,
    }
    
    df_page_i_1 = df_page_i_1.iloc[1:PARTYLIST_PAGE_I_1+1, :3].reset_index(drop=True)
    df_page_i_2 = df_page_i_2.iloc[1:PARTYLIST_PAGE_I_2+1, :3].reset_index(drop=True)
    df_page_i_3 = df_page_i_3.iloc[1:PARTYLIST_PAGE_I_3+1, :3].reset_index(drop=True)
    
    df = pd.concat([df_page_i_1, df_page_i_2, df_page_i_3])
    
    df.columns = ['หมายเลข', 'พรรค', 'คะแนน']
    df.drop(columns=['หมายเลข'], inplace=True)
    df.reset_index(drop=True)
    
    # เติมชื่อพรรค
    df["พรรค"] = PARTICIPANTS_PARTYLIST
    
    # Extract เฉพาะตัวเลขจากคะแนน
    df['คะแนน'] = (df['คะแนน']
                    .astype(str)
                    .str.replace(',', '')
                    .str.findall(r'\d+') 
                    .str[-1]
                    .pipe(pd.to_numeric, errors='coerce')
                    .fillna(0)
                    .astype(int))
    
    # เก็บผลลัพธ์
    save_ocr_output(file_name, page_num, summary, df)
    
    return summary, df

### Reader

In [ ]:
for file_config in OCR_FILES:
    total_pages = file_config["pages"]
    if KEYWORD in file_config["path"]:
        for page_num in range(1, total_pages, OCR_STEPS[KEYWORD]):
            read_partylist(file_config["path"], page_num)

# Check OCR Result

In [ ]:
dfs = []

for file_config in OCR_FILES:
    total_pages = file_config["pages"]
    if KEYWORD in file_config["path"]:
        for page_num in range(1, total_pages, OCR_STEPS[KEYWORD]):
            df_dict = load_data(file_config["path"], page_num)
            dfs.append(df_dict)

In [ ]:
dfs[0]